# Deep Learning — Module 2: Training & Optimization · Part 3
## Regularization — Preventing Overfitting

> Every concept in three layers: **Intuition → Math → Code**

---

## Table of Contents

| Section | Topic |
|---------|-------|
| **1** | Bias-Variance Tradeoff |
| **2** | L2 Regularization (Weight Decay) |
| **3** | L1 Regularization & Sparsity |
| **4** | Elastic Net (L1 + L2) |
| **5** | Dropout — Random Neuron Silencing |
| **6** | Inverted Dropout & Test-time Behaviour |
| **7** | Early Stopping |
| **8** | Weighted Loss (Class Imbalance) |
| **9** | PyTorch Regularization Demo |
| **10** | Master Interview Q&A Cheatsheet |


In [ ]:
# ── All imports ───────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
    "axes.prop_cycle": plt.cycler(color=[
        "#4C72B0","#DD8452","#55A868","#C44E52","#8172B3","#937860"
    ])
})
print("Imports ready ✓")


## 1. Bias-Variance Tradeoff — The Root of Overfitting

### Intuition: Fitting a Curve Through Points
Imagine fitting a curve to 10 noisy data points:
- **Underfit (High Bias)**: a straight line through 10 points — misses the real pattern
- **Overfit (High Variance)**: a degree-9 polynomial through 10 points — hits every point but wiggles wildly, fails on new data
- **Just right**: a gentle curve that captures the pattern but smooths over noise

**Bias** = how wrong the model is on average (systematic error)
**Variance** = how much the model changes when trained on different datasets (sensitivity to noise)

---

### Math: The Bias-Variance Decomposition

For a regression model, the expected prediction error at point $x$ decomposes as:

$$\mathbb{E}[(y - \hat{f}(x))^2] = \underbrace{(\mathbb{E}[\hat{f}(x)] - f(x))^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Irreducible noise}}$$

| Term | Meaning | Controlled by |
|------|---------|---------------|
| Bias² | How far the average prediction is from truth | Model capacity (too simple = high bias) |
| Variance | How much predictions vary across datasets | Model capacity (too complex = high variance) |
| σ² | Noise in the data itself | Cannot be reduced |

### The Tradeoff
- Increasing model complexity → Bias ↓, Variance ↑
- Decreasing model complexity → Bias ↑, Variance ↓

**Regularization = techniques to reduce variance without catastrophically increasing bias.**

### Diagnosing in Practice
| Symptom | Problem | Fix |
|---------|---------|-----|
| Train loss low, val loss high | High Variance (overfit) | Regularize, get more data |
| Both train & val loss high | High Bias (underfit) | Larger model, more epochs |
| Train ≈ val loss, both acceptable | Just right ✓ | — |


In [ ]:
# Figure 1: Bias-Variance Tradeoff — polynomial fitting example
np.random.seed(42)

# True function
x_true = np.linspace(0, 1, 200)
y_true = np.sin(2 * np.pi * x_true)

# Training data (noisy)
n_train = 12
x_train = np.linspace(0, 1, n_train)
y_train = np.sin(2 * np.pi * x_train) + np.random.randn(n_train) * 0.25

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))

degrees = [1, 4, 9]
titles  = ['Degree 1\n(Underfit — High Bias)', 'Degree 4\n(Just Right)',
           'Degree 9\n(Overfit — High Variance)']
colors  = ['#C44E52', '#55A868', '#8172B3']

for ax, deg, title, col in zip(axes[:3], degrees, titles, colors):
    p = np.polyfit(x_train, y_train, deg)
    y_pred = np.polyval(p, x_true)
    ax.plot(x_true, y_true, 'k--', lw=2, label='True f(x)', zorder=3)
    ax.plot(x_train, y_train, 'o', ms=7, color='#DD8452', label='Train data', zorder=4)
    ax.plot(x_true, y_pred, '-', lw=2.5, color=col, label=f'Fit (deg={deg})')
    ax.set_ylim(-2, 2); ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(title); ax.legend(fontsize=8)
    # Compute train & test MSE
    train_mse = np.mean((np.polyval(p, x_train) - y_train)**2)
    val_x = np.linspace(0, 1, 50)
    val_y = np.sin(2*np.pi*val_x) + np.random.randn(50)*0.25
    val_mse = np.mean((np.polyval(p, val_x) - val_y)**2)
    ax.text(0.02, -1.6, f'Train MSE: {train_mse:.3f}\nVal MSE:   {val_mse:.3f}',
            fontsize=9, color=col,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=col, alpha=0.8))

# ── Plot 4: Bias-Variance curves ──
deg_range = np.arange(1, 13)
train_errs, val_errs = [], []
for d in deg_range:
    p = np.polyfit(x_train, y_train, d)
    train_errs.append(np.mean((np.polyval(p, x_train) - y_train)**2))
    val_errs.append(np.mean((np.polyval(p, x_true) - y_true)**2))

axes[3].plot(deg_range, train_errs, 'o-', lw=2, color='#55A868', label='Train error (↓ bias)')
axes[3].plot(deg_range, val_errs,   's-', lw=2, color='#C44E52', label='Val error (↑ variance)')
axes[3].axvline(4, color='grey', ls='--', lw=1.5, label='Sweet spot')
axes[3].set_xlabel('Model complexity (polynomial degree)')
axes[3].set_ylabel('MSE'); axes[3].set_title('Bias-Variance Tradeoff Curve')
axes[3].legend(fontsize=9); axes[3].set_ylim(0, 0.5)

plt.suptitle('Figure 1: Bias-Variance Tradeoff', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 2. L2 Regularization (Weight Decay)

### Intuition: Penalise Large Weights
An overfit model has wildly large weights — it has learned to memorise noise by assigning huge importance to specific features.

**L2 regularization** adds a penalty to the loss for large weights, forcing the optimizer to keep them small. It's like telling the model: *"Find a good solution, but don't use extreme weights to get there."*

Think of it as a rubber band pulling all weights back toward zero.

---

### Math

**Modified loss:**
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{data}} + \frac{\lambda}{2}\sum_i w_i^2$$

**Gradient:**
$$\frac{\partial\mathcal{L}_{\text{total}}}{\partial w_i} = \frac{\partial\mathcal{L}_{\text{data}}}{\partial w_i} + \lambda w_i$$

**Update rule (SGD):**
$$w_i \leftarrow w_i - \eta\left(\frac{\partial\mathcal{L}_{\text{data}}}{\partial w_i} + \lambda w_i\right) = w_i(1 - \eta\lambda) - \eta\frac{\partial\mathcal{L}_{\text{data}}}{\partial w_i}$$

The $(1-\eta\lambda)$ factor = **weight decay**: every step, weights are multiplied by a value slightly less than 1, shrinking toward zero.

### What L2 Regularization Prefers Geometrically
Adding $\lambda\|w\|_2^2$ constraint = solutions close to the origin.
In 2D: the constraint region is a **circle** (L2 ball).
The regularised solution = intersection of loss contours with this circle.

### Effect on Solutions
| Without L2 | With L2 |
|-----------|---------|
| Weights can be arbitrarily large | Weights pushed toward 0 |
| Sensitive to input noise | Smoother, more robust predictions |
| All features potentially used | All features used, but with smaller coefficients |
| Sharp minima | Broader, flatter minima |

### Hyperparameter λ (weight decay)
- λ too small → no regularization effect
- λ too large → underfitting (all weights forced to 0)
- Typical values: `1e-4` to `1e-2`

### PyTorch
```python
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
# OR for Adam (less correct but common):
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
```

#### Interview Questions: L2
> **Q: Why is L2 regularization called "weight decay"?**
> A: Because the update rule multiplies weights by $(1-\eta\lambda) < 1$ each step — the weights literally decay toward zero. The name comes from pre-Adam optimizers where L2 reg and weight decay were equivalent.

> **Q: What distribution does L2 regularization implicitly assume for weights?**
> A: It corresponds to a **Gaussian prior** $p(w) = \mathcal{N}(0, 1/\lambda)$ on the weights. Maximising the MAP estimate with this prior gives exactly L2-regularised loss minimisation.

> **Q: Does L2 make weights exactly zero?**
> A: No — L2 shrinks weights toward zero but never exactly zero (the gradient $\lambda w$ approaches 0 as $w → 0$, so the push disappears). L1 can make weights exactly zero.


In [ ]:
# Figure 2: L2 Regularization — weight shrinkage, loss landscape, train vs val
np.random.seed(42)

# Synthetic regression with overfitting
N_train, N_val = 20, 100
x_tr  = np.random.uniform(0, 1, N_train)
y_tr  = np.sin(2*np.pi*x_tr) + np.random.randn(N_train) * 0.3
x_val = np.linspace(0, 1, N_val)
y_val = np.sin(2*np.pi*x_val)
x_plot = np.linspace(0, 1, 300)

deg = 10
poly = PolynomialFeatures(deg, include_bias=True)
X_tr   = poly.fit_transform(x_tr.reshape(-1,1))
X_val  = poly.transform(x_val.reshape(-1,1))
X_plot = poly.transform(x_plot.reshape(-1,1))

lambda_vals = [0, 1e-5, 1e-3, 1e-1, 10.0]
colors_l2   = ['#C44E52','#DD8452','#55A868','#4C72B0','#8172B3']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Plot 1: Fitted curves for different λ ──
axes[0].scatter(x_tr, y_tr, color='#DD8452', s=50, zorder=5, label='Train data')
axes[0].plot(x_plot, np.sin(2*np.pi*x_plot), 'k--', lw=2, label='True f(x)')
for lam, col in zip(lambda_vals, colors_l2):
    m = Ridge(alpha=lam, fit_intercept=False)
    m.fit(X_tr, y_tr)
    axes[0].plot(x_plot, m.predict(X_plot), color=col, lw=1.8, alpha=0.85,
                 label=f'λ={lam}')
axes[0].set_ylim(-2.5, 2.5); axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].set_title('L2 Ridge Regression (deg=10)'); axes[0].legend(fontsize=7.5)

# ── Plot 2: Weight magnitudes vs λ ──
weight_norms = []
for lam in lambda_vals:
    m = Ridge(alpha=lam, fit_intercept=False)
    m.fit(X_tr, y_tr)
    weight_norms.append(np.linalg.norm(m.coef_))

axes[1].semilogx([max(lam, 1e-6) for lam in lambda_vals], weight_norms,
                 'o-', lw=2.5, color='#4C72B0', ms=8)
axes[1].set_xlabel('λ (log scale)'); axes[1].set_ylabel('||w||₂ (weight norm)')
axes[1].set_title('L2 Shrinks Weights As λ Increases')
axes[1].set_xscale('log')

# ── Plot 3: Train vs Val loss vs λ ──
lam_range = np.logspace(-6, 2, 50)
tr_losses, val_losses = [], []
for lam in lam_range:
    m = Ridge(alpha=lam, fit_intercept=False)
    m.fit(X_tr, y_tr)
    tr_losses.append(np.mean((m.predict(X_tr) - y_tr)**2))
    val_losses.append(np.mean((m.predict(X_val) - y_val)**2))

axes[2].semilogx(lam_range, tr_losses,  lw=2.5, color='#55A868', label='Train MSE')
axes[2].semilogx(lam_range, val_losses, lw=2.5, color='#C44E52', label='Val MSE')
best_lam = lam_range[np.argmin(val_losses)]
axes[2].axvline(best_lam, color='grey', ls='--', lw=1.5, label=f'Best λ={best_lam:.1e}')
axes[2].set_xlabel('λ (log scale)'); axes[2].set_ylabel('MSE')
axes[2].set_title('Train vs Val Loss: Finding the Right λ'); axes[2].legend()

plt.suptitle('Figure 2: L2 Regularization Effect', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 3. L1 Regularization — Sparsity and Feature Selection

### Intuition: The Hard Cutoff
L2 gently pushes all weights toward zero. **L1 is more aggressive** — it applies a constant force regardless of weight magnitude, strong enough to push small weights all the way to exactly zero.

Imagine two parking scenarios:
- **L2**: a spring pulling your car toward the parking spot — force weakens as you get closer
- **L1**: a person physically pushing your car at constant force — they'll push you all the way to exactly the spot

Result: L1 produces **sparse solutions** — most weights are exactly zero, only the most important features have non-zero weights.

---

### Math

**Modified loss:**
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{data}} + \lambda\sum_i |w_i|$$

**Gradient (subgradient):**
$$\frac{\partial\mathcal{L}_{\text{total}}}{\partial w_i} = \frac{\partial\mathcal{L}_{\text{data}}}{\partial w_i} + \lambda\,\text{sign}(w_i)$$

The L1 gradient is **constant magnitude** $\lambda$ regardless of $|w_i|$:
- Even a tiny weight $w = 0.001$ gets pushed by force $\lambda$
- This is why it drives weights all the way to zero

**Geometric interpretation:** The L1 ball is a **diamond** (rhombus) shape.
Its corners lie on the axes. Loss contours tend to intersect these corners → exact zeros.

### L1 vs L2 Side by Side

| Property | L1 (Lasso) | L2 (Ridge) |
|----------|-----------|-----------|
| Penalty | $\lambda\|w\|_1 = \lambda\sum|w_i|$ | $\frac{\lambda}{2}\|w\|_2^2 = \frac{\lambda}{2}\sum w_i^2$ |
| Constraint region | Diamond (L1 ball) | Circle (L2 ball) |
| Produces exact zeros? | ✅ Yes | ❌ No (only shrinks) |
| Solution sparsity | ✅ Sparse | ❌ Dense |
| Robust to irrelevant features? | ✅ Eliminates them | ❌ Shrinks but keeps |
| Bayesian prior | Laplace $p(w) \propto e^{-\lambda|w|}$ | Gaussian $p(w) \propto e^{-\lambda w^2}$ |
| Differentiable? | ❌ Not at 0 | ✅ Yes |
| Best for | Feature selection, sparse problems | General regularization |

### When to use L1
- NLP text models (most words are irrelevant → sparsity useful)
- Genomics (thousands of genes, few causal → sparsity critical)
- Any problem where you suspect most features are noise

#### Interview Questions: L1
> **Q: Why does L1 produce sparse solutions but L2 doesn't?**
> A: Geometrically: the L1 ball (diamond) has corners positioned on coordinate axes. Loss contours tend to touch the diamond at corners, setting some weights exactly to 0. The L2 ball (circle) has no corners — loss contours touch it smoothly, shrinking all weights but never reaching 0.

> **Q: What is the subgradient of L1 at w=0?**
> A: The subgradient at 0 is any value in $[-\lambda, \lambda]$. In practice, we use 0 as the subgradient, meaning a weight at 0 stays at 0.

> **Q: What Bayesian prior does L1 regularization correspond to?**
> A: A Laplace (double exponential) prior $p(w) \propto e^{-\lambda|w|}$ — heavier tails than Gaussian, more mass at zero, fewer large values.


In [ ]:
# Figure 3: L1 Regularization — sparsity, diamond constraint, L1 vs L2
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Plot 1: L1 vs L2 constraint regions ──
theta = np.linspace(0, 2*np.pi, 500)
# L2 ball (circle)
axes[0].plot(np.cos(theta), np.sin(theta), color='#4C72B0', lw=2.5, label='L2 ball (circle)')
# L1 ball (diamond)
axes[0].plot([1,0,-1,0,1], [0,1,0,-1,0], color='#C44E52', lw=2.5, label='L1 ball (diamond)')
# Loss contour ellipses
for r in [1.2, 1.6, 2.0]:
    ell_x = r * np.cos(theta) + 1.5
    ell_y = 0.5 * r * np.sin(theta) + 0.5
    axes[0].plot(ell_x, ell_y, color='#55A868', lw=1, alpha=0.5)
axes[0].annotate('L1 solution\n(corner = sparse!)', xy=(1,0), xytext=(1.4, 0.6),
                 fontsize=9, color='#C44E52',
                 arrowprops=dict(arrowstyle='->', color='#C44E52'))
axes[0].annotate('L2 solution\n(smooth edge = dense)', xy=(0.71,0.63), xytext=(-1.4, 1.1),
                 fontsize=9, color='#4C72B0',
                 arrowprops=dict(arrowstyle='->', color='#4C72B0'))
axes[0].set_xlim(-2.2, 2.5); axes[0].set_ylim(-1.8, 1.8)
axes[0].set_xlabel('w₁'); axes[0].set_ylabel('w₂')
axes[0].set_title('L1 Ball (Diamond) vs L2 Ball (Circle)'); axes[0].legend(); axes[0].set_aspect('equal')

# ── Plot 2: Sparsity — weight histograms ──
lam_l1, lam_l2 = 0.01, 0.01
m_l1 = Lasso(alpha=lam_l1,  fit_intercept=False, max_iter=5000)
m_l2 = Ridge(alpha=lam_l2*N_train, fit_intercept=False)
m_l1.fit(X_tr, y_tr); m_l2.fit(X_tr, y_tr)

axes[1].hist(np.abs(m_l2.coef_), bins=20, color='#4C72B0', alpha=0.7, label=f'L2 (Ridge) λ={lam_l2}')
axes[1].hist(np.abs(m_l1.coef_), bins=20, color='#C44E52', alpha=0.7, label=f'L1 (Lasso) λ={lam_l1}')
axes[1].set_xlabel('|weight|'); axes[1].set_ylabel('Count')
axes[1].set_title('Weight Magnitude Distribution')
axes[1].legend()
n_zero_l1 = np.sum(np.abs(m_l1.coef_) < 1e-6)
n_zero_l2 = np.sum(np.abs(m_l2.coef_) < 1e-6)
axes[1].text(0.5, 0.75, f'L1 exact zeros: {n_zero_l1}/{len(m_l1.coef_)}
L2 exact zeros: {n_zero_l2}/{len(m_l2.coef_)}',
             transform=axes[1].transAxes, fontsize=10,
             bbox=dict(boxstyle='round', facecolor='white', edgecolor='grey'))

# ── Plot 3: L1 fitted curves vs L2 ──
axes[2].scatter(x_tr, y_tr, color='#DD8452', s=50, zorder=5, label='Train data')
axes[2].plot(x_plot, np.sin(2*np.pi*x_plot), 'k--', lw=2, label='True f(x)')
axes[2].plot(x_plot, m_l1.predict(X_plot), color='#C44E52', lw=2.5, label=f'L1 (Lasso)')
axes[2].plot(x_plot, m_l2.predict(X_plot), color='#4C72B0', lw=2.5, label=f'L2 (Ridge)')

no_reg = LinearRegression(fit_intercept=False).fit(X_tr, y_tr)
axes[2].plot(x_plot, no_reg.predict(X_plot), color='#937860', lw=1.8, ls='--', label='No reg')
axes[2].set_ylim(-2.5, 2.5); axes[2].set_xlabel('x'); axes[2].set_ylabel('y')
axes[2].set_title('L1 vs L2 Fitted Curves'); axes[2].legend(fontsize=9)

plt.suptitle('Figure 3: L1 Regularization — Sparsity & Diamond Constraint', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"L1 (Lasso) — weights exactly zero: {n_zero_l1}/{len(m_l1.coef_)}")
print(f"L2 (Ridge) — weights exactly zero: {n_zero_l2}/{len(m_l2.coef_)}")


## 4. Elastic Net — The Best of L1 and L2

### Intuition
L1 is too aggressive → can remove useful correlated features
L2 is too soft → won't eliminate irrelevant features

**Elastic Net** combines both penalties with a mixing parameter $\alpha$:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{data}} + \lambda\left[\alpha\|w\|_1 + \frac{(1-\alpha)}{2}\|w\|_2^2\right]$$

- $\alpha = 0$ → pure L2 (Ridge)
- $\alpha = 1$ → pure L1 (Lasso)
- $0 < \alpha < 1$ → Elastic Net

### Key Advantage: Grouped Selection
When features are correlated (e.g., word embeddings, pixel patches), Lasso arbitrarily picks one from a correlated group. Elastic Net tends to select all correlated features together or none — more stable for correlated feature sets.

### In Deep Learning
L1 is rarely used directly in neural networks (gradient discontinuity at 0 causes issues with autograd).
Elastic Net is similarly uncommon in deep learning.

**In deep learning we use:**
- **L2 / Weight Decay** (common, built into AdamW)
- **Dropout** (more effective for neural networks)
- **Batch Normalization** (has implicit regularization effect)

Elastic Net is most relevant for **linear/shallow models** and classical ML.

#### Interview Questions: Elastic Net
> **Q: When would you choose Elastic Net over pure L1 or L2?**
> A: When you have correlated features — Lasso unstably picks one, Elastic Net handles groups. Also when Lasso gives too sparse a solution (important features eliminated) but you still want some sparsity.

> **Q: Why is L1 regularization less common in deep learning than L2?**
> A: The subgradient at 0 complicates autograd. More practically, dropout and batch normalisation are more effective for neural network regularisation. L2 (via AdamW weight decay) is the standard for deep learning.


In [ ]:
# Figure 4: L1, L2, Elastic Net comparison
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── Plot 1: Weight profiles for different regularizers ──
lam = 0.005
models = {
    'No Reg':          LinearRegression(fit_intercept=False).fit(X_tr, y_tr),
    'L2 (Ridge)':      Ridge(alpha=lam*N_train, fit_intercept=False).fit(X_tr, y_tr),
    'L1 (Lasso)':      Lasso(alpha=lam, fit_intercept=False, max_iter=5000).fit(X_tr, y_tr),
    'Elastic (α=0.5)': ElasticNet(alpha=lam, l1_ratio=0.5, fit_intercept=False, max_iter=5000).fit(X_tr, y_tr),
}
colors_en = ['#937860','#4C72B0','#C44E52','#8172B3']
x_idx = np.arange(len(list(models.values())[0].coef_))

for (name, m), col in zip(models.items(), colors_en):
    axes[0].plot(x_idx, m.coef_, 'o-', ms=5, lw=1.8, color=col, label=name, alpha=0.85)
axes[0].axhline(0, color='grey', ls='--', lw=1)
axes[0].set_xlabel('Coefficient index'); axes[0].set_ylabel('Weight value')
axes[0].set_title('Weight Profiles: All Regularizers'); axes[0].legend(fontsize=9)

# ── Plot 2: Sparsity vs λ ──
lam_range = np.logspace(-4, 0, 40)
sparsity_l1, sparsity_en = [], []
for lam in lam_range:
    ml1 = Lasso(alpha=lam, fit_intercept=False, max_iter=5000).fit(X_tr, y_tr)
    men = ElasticNet(alpha=lam, l1_ratio=0.5, fit_intercept=False, max_iter=5000).fit(X_tr, y_tr)
    sparsity_l1.append(np.sum(np.abs(ml1.coef_) < 1e-6))
    sparsity_en.append(np.sum(np.abs(men.coef_) < 1e-6))

axes[1].semilogx(lam_range, sparsity_l1, lw=2.5, color='#C44E52', label='L1 (Lasso) — sparse')
axes[1].semilogx(lam_range, sparsity_en, lw=2.5, color='#8172B3', label='Elastic Net α=0.5')
axes[1].set_xlabel('λ (log scale)'); axes[1].set_ylabel('Number of zero weights')
axes[1].set_title('Sparsity vs Regularization Strength'); axes[1].legend()

plt.suptitle('Figure 4: Elastic Net = L1 + L2', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 5. Dropout — Random Neuron Silencing

### Intuition: Don't Rely on Any Single Neuron
Imagine a company where every decision depends on one "star employee."
If that employee is absent, everything breaks down — the company is fragile.

A resilient company trains every employee to handle multiple tasks.
If anyone is absent, others can compensate.

Dropout says exactly this to your neural network: *"Don't rely on any single neuron — they might be randomly absent."*

During training, each neuron is **randomly set to 0** with probability $p$ (the dropout rate).
This forces the network to learn **redundant representations** — multiple neurons must encode the same information.

### Why It Works (the ensemble interpretation)
A network with $n$ neurons has $2^n$ possible subnetworks (each neuron on/off).
With dropout, each training step trains a different subnetwork.
At test time, the full network = an approximate **ensemble average** of all $2^n$ subnetworks.

---

### Math

**During training** (for each forward pass):
$$r_j^{(l)} \sim \text{Bernoulli}(1 - p) \quad \text{(1 = keep, 0 = drop)}$$
$$\tilde{a}^{(l)} = r^{(l)} \odot a^{(l)}$$

- Each neuron independently dropped with probability $p$
- Dropped neurons output 0 and **receive no gradient**

**Dropout rates by layer type:**
| Layer | Typical dropout rate |
|-------|---------------------|
| Fully connected (hidden) | 0.5 |
| Fully connected (input) | 0.2 |
| Convolutional | 0.1–0.2 (or none) |
| Transformer attention | 0.1 |

### The Dropout Mask
The mask $r^{(l)}$ is **resampled at every forward pass** during training.
This is key — not a fixed subset dropped, but random each time.


In [ ]:
# Figure 5: Dropout — masking demo, ensemble interpretation
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Plot 1: Dropout mask visualisation ──
n_neurons, n_passes = 10, 8
masks = np.random.binomial(1, 0.5, (n_passes, n_neurons))
im = axes[0].imshow(masks, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[0].set_xlabel('Neuron index'); axes[0].set_ylabel('Training step')
axes[0].set_title('Dropout Mask (green=kept, red=dropped, p=0.5)')
axes[0].set_xticks(range(n_neurons)); axes[0].set_yticks(range(n_passes))
axes[0].set_xticklabels([f'N{i}' for i in range(n_neurons)], fontsize=8)
plt.colorbar(im, ax=axes[0], ticks=[0,1], label='Active')

# ── Plot 2: Effect of dropout on activation values ──
np.random.seed(5)
activations = np.abs(np.random.randn(50)) + 0.5
p_drop = 0.5
mask = np.random.binomial(1, 1-p_drop, 50)
dropped = activations * mask

x_idx = np.arange(50)
axes[1].bar(x_idx, activations, color='#4C72B0', alpha=0.6, label='Original activations')
axes[1].bar(x_idx, dropped,     color='#C44E52', alpha=0.9, label=f'After dropout (p={p_drop})',
            width=0.5)
axes[1].set_xlabel('Neuron index'); axes[1].set_ylabel('Activation value')
axes[1].set_title(f'Dropout p={p_drop}: {mask.sum()}/50 neurons kept')
axes[1].legend(fontsize=9)

# ── Plot 3: Overfitting with vs without dropout ──
torch.manual_seed(0)
N = 60
X_data = torch.linspace(-3, 3, N).unsqueeze(1)
y_data = torch.sin(X_data) + 0.3 * torch.randn_like(X_data)

class Net(nn.Module):
    def __init__(self, use_dropout=False, p=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Dropout(p) if use_dropout else nn.Identity(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Dropout(p) if use_dropout else nn.Identity(),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x)

def train_net(use_dropout, epochs=1500):
    net = Net(use_dropout=use_dropout)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for _ in range(epochs):
        net.train()
        opt.zero_grad()
        loss = nn.MSELoss()(net(X_data), y_data)
        loss.backward()
        opt.step()
    return net

net_nodrop = train_net(False)
net_drop   = train_net(True)

x_test = torch.linspace(-3.5, 3.5, 200).unsqueeze(1)
net_nodrop.eval(); net_drop.eval()
with torch.no_grad():
    y_nodrop = net_nodrop(x_test).numpy()
    y_drop   = net_drop(x_test).numpy()

axes[2].scatter(X_data.numpy(), y_data.numpy(), color='#DD8452', s=25, zorder=4, label='Data', alpha=0.7)
axes[2].plot(x_test.numpy(), np.sin(x_test.numpy()), 'k--', lw=2, label='True sin(x)')
axes[2].plot(x_test.numpy(), y_nodrop, color='#C44E52', lw=2.5, label='No Dropout (overfit)')
axes[2].plot(x_test.numpy(), y_drop,   color='#55A868', lw=2.5, label='Dropout p=0.4')
axes[2].set_ylim(-2, 2); axes[2].set_xlabel('x'); axes[2].set_ylabel('y')
axes[2].set_title('Dropout Reduces Overfitting'); axes[2].legend(fontsize=9)

plt.suptitle('Figure 5: Dropout — Random Neuron Silencing', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 6. Inverted Dropout & Test-time Behaviour

### The Train vs Test Discrepancy
At training time: neurons are randomly dropped → expected output of a layer is reduced.
At test time: all neurons are active → output is larger than training average.

Without correction, the **expected activation at test time ≠ expected activation at train time**.
This breaks the model — test outputs would be systematically larger.

### Two Solutions

#### Solution 1: Scale at Test Time (Vanilla Dropout)
- Train: drop neurons with probability $p$
- Test: **multiply all activations by $(1-p)$** to match training expectation

Problem: every deployment must remember to scale. Easy to forget.

#### Solution 2: Inverted Dropout (Standard in PyTorch)
- Train: drop neurons AND **divide surviving activations by $(1-p)$** to compensate immediately
- Test: **do nothing** — no scaling needed

$$\tilde{a}^{(l)} = \frac{r^{(l)} \odot a^{(l)}}{1-p}$$

**This is the standard implementation** — PyTorch's `nn.Dropout` uses inverted dropout.

### Why Inverted Dropout is Better
- Test-time inference is simpler (no scaling step)
- Easier to implement correctly (no separate test-time code path)
- The network learns to operate with the already-scaled activations

### Critical: `model.train()` vs `model.eval()`
```python
model.train()   # Dropout IS active (neurons randomly dropped)
model.eval()    # Dropout IS NOT active (all neurons used, no scaling needed)
```

**Always call `model.eval()` before inference!**
Forgetting this is one of the most common bugs in PyTorch code.

#### Interview Questions: Inverted Dropout
> **Q: What is the difference between vanilla dropout and inverted dropout?**
> A: Vanilla dropout scales activations at test time (multiply by 1-p). Inverted dropout scales during training (divide by 1-p), requiring no test-time changes. PyTorch uses inverted dropout — `nn.Dropout` does nothing at eval time.

> **Q: Why must you call model.eval() before inference?**
> A: Dropout (and BatchNorm) behave differently at train vs test time. Forgetting `model.eval()` means dropout is still randomly dropping neurons during inference — your predictions will be stochastic and incorrect.

> **Q: What is the expected output of a dropout layer?**
> A: With inverted dropout: $\mathbb{E}[\tilde{a}] = (1-p) \cdot \frac{a}{1-p} = a$. The expected output equals the input — the network's expected behaviour is identical at train and test time, which is why inverted dropout works without test-time scaling.


In [ ]:
# Figure 6: Inverted Dropout — train vs test, PyTorch model.train()/eval()
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Plot 1: Vanilla vs Inverted dropout expectations ──
p = 0.5
activations = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
N = 10000

# Vanilla dropout (train): E[output] = (1-p) * input
vanilla_train = []
for _ in range(N):
    mask = (torch.rand_like(activations) > p).float()
    vanilla_train.append((mask * activations).mean().item())

# Inverted dropout (train): E[output] = input
inverted_train = []
for _ in range(N):
    mask = (torch.rand_like(activations) > p).float()
    inverted_train.append((mask * activations / (1-p)).mean().item())

true_mean = activations.mean().item()

axes[0].hist(vanilla_train,  bins=40, color='#C44E52', alpha=0.7,
             label=f'Vanilla (mean={np.mean(vanilla_train):.2f})')
axes[0].hist(inverted_train, bins=40, color='#55A868', alpha=0.7,
             label=f'Inverted (mean={np.mean(inverted_train):.2f})')
axes[0].axvline(true_mean, color='k', ls='--', lw=2, label=f'True mean={true_mean:.2f}')
axes[0].set_xlabel('Layer output mean'); axes[0].set_ylabel('Frequency')
axes[0].set_title('Train-time: Inverted Dropout Preserves Mean')
axes[0].legend(fontsize=9)

# ── Plot 2: model.train() vs model.eval() ──
net = nn.Sequential(nn.Linear(5, 5), nn.Dropout(p=0.5))
x_in = torch.ones(100, 5)

net.train()
outputs_train = [net(x_in).detach().mean().item() for _ in range(50)]

net.eval()
outputs_eval  = [net(x_in).detach().mean().item() for _ in range(50)]

axes[1].plot(outputs_train, color='#C44E52', lw=1.5, label='model.train() — stochastic')
axes[1].plot(outputs_eval,  color='#55A868', lw=1.5, label='model.eval() — deterministic')
axes[1].set_xlabel('Forward pass #'); axes[1].set_ylabel('Output mean')
axes[1].set_title('model.train() vs model.eval() — Dropout Active/Inactive')
axes[1].legend()

# ── Plot 3: Variance at test time (the bug) ──
torch.manual_seed(0)
class BigNet(nn.Module):
    def __init__(self): super().__init__(); self.net = nn.Sequential(
        nn.Linear(10, 64), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(64, 1))
    def forward(self, x): return self.net(x)

bignet = BigNet()
x_test = torch.randn(1, 10)

# Forget eval — preds are random
bignet.train()
wrong_preds = [bignet(x_test).item() for _ in range(200)]
# Correct
bignet.eval()
with torch.no_grad():
    correct_pred = bignet(x_test).item()

axes[2].hist(wrong_preds, bins=30, color='#C44E52', alpha=0.85,
             label=f'model.train() — std={np.std(wrong_preds):.3f}')
axes[2].axvline(correct_pred, color='#55A868', lw=3,
                label=f'model.eval() = {correct_pred:.3f}')
axes[2].set_xlabel('Prediction value'); axes[2].set_ylabel('Count')
axes[2].set_title('Bug: Forgetting model.eval() → Random Predictions!')
axes[2].legend(fontsize=9)

plt.suptitle('Figure 6: Inverted Dropout & Train/Eval Mode', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Key rule:")
print("  model.train()  → use during training   (dropout ON)")
print("  model.eval()   → use during inference  (dropout OFF)")
print("  Forgetting model.eval() is one of the most common PyTorch bugs!")


## 7. Early Stopping

### Intuition: Stop Before You Memorise
Early stopping watches the **validation loss** during training.
When it stops improving (and starts getting worse), training is stopped — before the model memorises the training set.

Think of it as a student who stops studying when their practice test score peaks.
More cramming after that point hurts performance on the real exam.

---

### Algorithm

```
best_val_loss  = ∞
patience_count = 0
patience       = 10  # how many epochs to wait

for epoch in range(max_epochs):
    train_one_epoch()
    val_loss = evaluate()

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        patience_count = 0
        save_checkpoint()        # save best model weights!
    else:
        patience_count += 1

    if patience_count >= patience:
        break                    # stop training

load_checkpoint()  # restore best weights
```

### Key Parameters
| Parameter | Effect |
|-----------|--------|
| `patience` | How many epochs to wait for improvement before stopping. Too small → stop prematurely. Too large → wastes compute. Typical: 5–20. |
| `min_delta` | Minimum improvement to count as improvement. Prevents stopping when val loss plateaus at noise level. |
| `restore_best_weights` | Always do this — stop when patience exhausted but use the checkpoint from the *best* epoch, not the last. |

### Why Validation Loss Matters (not Training Loss)
Training loss always decreases with more epochs (the model can always memorise more).
Validation loss is the true signal: it drops while the model generalises, then rises as it overfits.

### Early Stopping = Implicit Regularization
Early stopping implicitly constrains the model from fully converging on training data.
Equivalent (approximately) to L2 regularization in some settings (Goodfellow et al., 2016).

#### Interview Questions: Early Stopping
> **Q: What is the patience hyperparameter in early stopping?**
> A: Number of epochs to wait without improvement before stopping. Acts as a delay to avoid stopping on temporary plateaus. Larger patience = more training, better final convergence if training is noisy.

> **Q: Why should you restore best weights when using early stopping?**
> A: Training continues for `patience` epochs after the best val loss. In those extra epochs, the model is overfitting. The best weights were at the peak validation performance, not at the stop point.

> **Q: Early stopping vs L2 regularization — do you need both?**
> A: Both are complementary. Use L2 to prevent individual weights from growing too large. Use early stopping to prevent the model from overfitting the training set in aggregate. In practice, use both together.


In [ ]:
# Figure 7: Early Stopping — train vs val curves, patience mechanism
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

epochs = np.arange(1, 151)

# Simulate training/val curves
train_loss = 1.5 * np.exp(-epochs * 0.04) + 0.08 + 0.005*np.random.randn(150)
val_loss   = 1.5 * np.exp(-epochs * 0.035) + 0.15

# Val loss starts increasing after epoch ~60 (overfitting)
overfit_mask = epochs > 60
val_loss[overfit_mask] = (0.22 + 0.002*(epochs[overfit_mask]-60)
                          + 0.01*np.random.randn(overfit_mask.sum()))

# Best val epoch
best_epoch = np.argmin(val_loss) + 1
patience   = 15
stop_epoch = min(best_epoch + patience, 150)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ── Plot 1: Loss curves with annotations ──
axes[0].plot(epochs, train_loss, color='#4C72B0', lw=2, label='Training loss')
axes[0].plot(epochs, val_loss,   color='#C44E52', lw=2, label='Validation loss')
axes[0].axvline(best_epoch, color='#55A868', ls='--', lw=2,
                label=f'Best val loss (epoch {best_epoch})')
axes[0].axvline(stop_epoch, color='#DD8452', ls='--', lw=2,
                label=f'Early stop (epoch {stop_epoch}, patience={patience})')
axes[0].axvspan(best_epoch, stop_epoch, alpha=0.08, color='#DD8452', label='Waiting (patience)')
axes[0].scatter([best_epoch], [val_loss[best_epoch-1]], s=150, color='#55A868', zorder=5)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Early Stopping: Train vs Validation Loss')
axes[0].legend(fontsize=9)

# Annotation
axes[0].annotate('Overfitting starts here →
Val loss rises, train still falls',
                 xy=(70, val_loss[69]), xytext=(85, 0.5),
                 fontsize=8.5, color='#C44E52',
                 arrowprops=dict(arrowstyle='->', color='#C44E52'))

# ── Plot 2: Patience counter ──
patience_counter = np.zeros(stop_epoch)
best_so_far = val_loss[0]
counter = 0
for i in range(stop_epoch):
    if val_loss[i] < best_so_far:
        best_so_far = val_loss[i]
        counter = 0
    else:
        counter += 1
    patience_counter[i] = counter

axes[1].fill_between(np.arange(1, stop_epoch+1), patience_counter,
                     color='#DD8452', alpha=0.4, label='Patience counter')
axes[1].plot(np.arange(1, stop_epoch+1), patience_counter,
             color='#DD8452', lw=2)
axes[1].axhline(patience, color='#C44E52', ls='--', lw=2.5,
                label=f'Patience limit = {patience}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Patience counter')
axes[1].set_title('Patience Counter — Triggers at Limit')
axes[1].legend()

plt.suptitle('Figure 7: Early Stopping', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"Best epoch: {best_epoch} (val_loss={val_loss[best_epoch-1]:.4f})")
print(f"Stop epoch: {stop_epoch} (patience={patience})")
print(f"Saved {150 - stop_epoch} wasted epochs → restore weights from epoch {best_epoch}")


## 8. Weighted Loss — Handling Class Imbalance

### Intuition: The Rare Disease Problem
Suppose you're classifying medical images: 95% healthy, 5% cancerous.
A model that always predicts "healthy" achieves 95% accuracy — and is completely useless.

The problem: the model sees 19× more healthy examples, so it optimises heavily for them.
The rare class (cancer) barely influences the loss.

**Weighted loss** gives the rare class a higher weight in the loss function — making each rare example count more.

---

### Math

**Weighted Cross-Entropy:**
$$\mathcal{L}_{\text{weighted}} = -\frac{1}{n}\sum_{i=1}^{n} w_{y_i} \cdot \log(\hat{y}_{i, y_i})$$

Where $w_k$ = weight for class $k$.

**Common weighting strategy — inverse frequency:**
$$w_k = \frac{n}{K \cdot n_k}$$

Where $n_k$ = number of samples in class $k$, $K$ = number of classes.

Class with 5% samples gets weight $= 1/0.05 = 20$.
Class with 95% samples gets weight $= 1/0.95 \approx 1.05$.

**PyTorch:**
```python
# Compute class weights
class_counts = torch.tensor([950., 50.])     # 950 class-0, 50 class-1
weights = class_counts.sum() / (len(class_counts) * class_counts)
criterion = nn.CrossEntropyLoss(weight=weights)
```

### Other Techniques for Class Imbalance
| Technique | How | When |
|-----------|-----|------|
| **Weighted Loss** | Increase loss weight for minority class | General purpose |
| **Oversampling (SMOTE)** | Generate synthetic minority samples | Small datasets |
| **Undersampling** | Remove majority class samples | Large datasets |
| **Focal Loss** | Down-weight easy examples | Object detection (covered with YOLO) |
| **Class-balanced sampling** | Sample equal batches per class | Training loop |

#### Interview Questions: Weighted Loss
> **Q: How do you handle class imbalance in a binary classification problem?**
> A: Multiple approaches: (1) Weighted loss with inverse-frequency weights. (2) Oversampling minority class (SMOTE). (3) Undersampling majority. (4) Use F1/AUC-ROC as metric instead of accuracy. (5) For severe imbalance, use Focal Loss. Choice depends on dataset size and severity of imbalance.

> **Q: What metric should you use when classes are imbalanced?**
> A: Never accuracy alone. Use: F1-score (harmonic mean of precision/recall), AUC-ROC (ranking quality), PR-AUC (better for severe imbalance than ROC). The choice depends on the relative cost of false positives vs false negatives.


In [ ]:
# Figure 8: Weighted Loss — class imbalance effect
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# Imbalanced binary dataset (90% class 0, 10% class 1)
N = 500
n_class0 = int(N * 0.90)
n_class1 = N - n_class0

X0 = torch.randn(n_class0, 2) + torch.tensor([0.0, 0.0])
X1 = torch.randn(n_class1, 2) + torch.tensor([2.0, 2.0])
X  = torch.cat([X0, X1])
y  = torch.cat([torch.zeros(n_class0), torch.ones(n_class1)]).long()

print(f"Class distribution: {n_class0} class-0 ({n_class0/N*100:.0f}%), "
      f"{n_class1} class-1 ({n_class1/N*100:.0f}%)")

class SimpleNet(nn.Module):
    def __init__(self): super().__init__(); self.l = nn.Linear(2, 2)
    def forward(self, x): return self.l(x)

def train_model(use_weights, epochs=300):
    net = SimpleNet()
    if use_weights:
        counts = torch.bincount(y).float()
        weights = counts.sum() / (len(counts) * counts)
        criterion = nn.CrossEntropyLoss(weight=weights)
    else:
        criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(net.parameters(), lr=0.02)
    for _ in range(epochs):
        opt.zero_grad()
        criterion(net(X), y).backward()
        opt.step()
    with torch.no_grad():
        preds = net(X).argmax(dim=1)
    return preds.numpy()

preds_unweighted = train_model(False)
preds_weighted   = train_model(True)

def metrics(preds, y_np):
    tp = ((preds==1) & (y_np==1)).sum()
    fp = ((preds==1) & (y_np==0)).sum()
    fn = ((preds==0) & (y_np==1)).sum()
    tn = ((preds==0) & (y_np==0)).sum()
    acc = (tp+tn)/(tp+fp+fn+tn)
    prec = tp/(tp+fp) if (tp+fp)>0 else 0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
    return {'Accuracy':acc, 'Precision':prec, 'Recall':rec, 'F1':f1}

y_np = y.numpy()
m_unw = metrics(preds_unweighted, y_np)
m_w   = metrics(preds_weighted,   y_np)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── Plot 1: Metrics bar chart ──
metric_names = list(m_unw.keys())
x = np.arange(len(metric_names))
w = 0.35
axes[0].bar(x-w/2, list(m_unw.values()), w, label='No weighting', color='#C44E52', alpha=0.88)
axes[0].bar(x+w/2, list(m_w.values()),   w, label='Weighted loss', color='#55A868', alpha=0.88)
axes[0].set_xticks(x); axes[0].set_xticklabels(metric_names)
axes[0].set_ylabel('Score'); axes[0].set_ylim(0, 1.1)
axes[0].set_title('Effect of Weighted Loss on Metrics')
axes[0].legend()
for i, (u, w_val) in enumerate(zip(m_unw.values(), m_w.values())):
    axes[0].text(i-w/2, u+0.02, f'{u:.2f}', ha='center', fontsize=9, color='#C44E52')
    axes[0].text(i+w/2, w_val+0.02, f'{w_val:.2f}', ha='center', fontsize=9, color='#55A868')

# ── Plot 2: Decision boundary comparison ──
xx, yy = np.meshgrid(np.linspace(-4,6,150), np.linspace(-4,6,150))
grid_t = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

nets = {}
for use_w, name in [(False,'Unweighted'), (True,'Weighted')]:
    net = SimpleNet()
    counts = torch.bincount(y).float()
    weights = counts.sum() / (len(counts) * counts)
    crit = nn.CrossEntropyLoss(weight=weights if use_w else None)
    opt  = torch.optim.Adam(net.parameters(), lr=0.02)
    for _ in range(300):
        opt.zero_grad(); crit(net(X), y).backward(); opt.step()
    nets[name] = net

# Plot weighted boundary only  
with torch.no_grad():
    probs_unw = torch.softmax(nets['Unweighted'](grid_t), dim=1)[:,1].numpy().reshape(xx.shape)
    probs_w   = torch.softmax(nets['Weighted'](grid_t), dim=1)[:,1].numpy().reshape(xx.shape)

axes[1].contourf(xx, yy, probs_w, levels=20, cmap='RdBu', alpha=0.7)
axes[1].contour(xx, yy, probs_unw, levels=[0.5], colors='#C44E52', linewidths=2.5,
                linestyles='--')
axes[1].contour(xx, yy, probs_w, levels=[0.5], colors='#55A868', linewidths=2.5)
axes[1].scatter(X0[:,0], X0[:,1], c='#4C72B0', s=10, alpha=0.5, label='Class 0 (90%)')
axes[1].scatter(X1[:,0], X1[:,1], c='#C44E52', s=40, zorder=5, label='Class 1 (10%)')
from matplotlib.lines import Line2D
axes[1].legend(handles=[
    Line2D([0],[0],color='#C44E52',ls='--',lw=2,label='Boundary (unweighted)'),
    Line2D([0],[0],color='#55A868',lw=2,label='Boundary (weighted)'),
    Line2D([0],[0],color='#4C72B0',marker='o',ls='',label='Class 0'),
    Line2D([0],[0],color='#C44E52',marker='o',ls='',label='Class 1'),
], fontsize=8)
axes[1].set_title('Decision Boundary Shift with Class Weighting')

plt.suptitle('Figure 8: Weighted Loss — Handling Class Imbalance', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---

## 9. PyTorch Regularization Demo — All Techniques Together

Training the same network with different regularization strategies to compare their effect.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# ── Dataset: small noisy sin regression ───────────────────────────
N_tr, N_val = 30, 200
X_tr  = torch.linspace(-3, 3, N_tr).unsqueeze(1)
y_tr  = torch.sin(X_tr) + 0.4*torch.randn_like(X_tr)
X_val = torch.linspace(-3, 3, N_val).unsqueeze(1)
y_val = torch.sin(X_val)

# ── Base model ────────────────────────────────────────────────────
def make_model(dropout_p=0.0):
    return nn.Sequential(
        nn.Linear(1, 128), nn.ReLU(),
        nn.Dropout(dropout_p),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Dropout(dropout_p),
        nn.Linear(128, 1)
    )

# ── Training function ─────────────────────────────────────────────
def train(model, opt, epochs=1000):
    crit = nn.MSELoss()
    tr_losses, val_losses = [], []
    best_val, best_ep, patience, counter = 1e9, 0, 50, 0

    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        loss = crit(model(X_tr), y_tr)
        loss.backward()
        opt.step()
        tr_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_l = crit(model(X_val), y_val).item()
        val_losses.append(val_l)

        # Early stopping
        if val_l < best_val - 1e-5:
            best_val = val_l
            best_ep = ep
            counter = 0
            best_state = {k: v.clone() for k,v in model.state_dict().items()}
        else:
            counter += 1
        if counter >= patience:
            break

    model.load_state_dict(best_state)
    return tr_losses, val_losses, best_ep

experiments = {
    'No regularization':  (make_model(0.0),   optim.Adam(make_model(0.0).parameters(), lr=1e-3)),
    'L2 (weight_decay)':  (make_model(0.0),   None),
    'Dropout (p=0.3)':    (make_model(0.3),   None),
    'L2 + Dropout':       (make_model(0.3),   None),
}
# Recreate with proper opts (needed because model must match opt)
models_opts = [
    ('No regularization', make_model(0.0), lambda m: optim.Adam(m.parameters(), lr=1e-3, weight_decay=0)),
    ('L2 (wd=0.01)',      make_model(0.0), lambda m: optim.AdamW(m.parameters(), lr=1e-3, weight_decay=0.01)),
    ('Dropout p=0.3',     make_model(0.3), lambda m: optim.Adam(m.parameters(), lr=1e-3, weight_decay=0)),
    ('L2 + Dropout',      make_model(0.3), lambda m: optim.AdamW(m.parameters(), lr=1e-3, weight_decay=0.01)),
]

colors  = ['#C44E52','#4C72B0','#55A868','#8172B3']
results = {}
for name, model, opt_fn in models_opts:
    opt = opt_fn(model)
    tr, val, best_ep = train(model, opt, epochs=1500)
    results[name] = {'tr': tr, 'val': val, 'model': model, 'best_ep': best_ep}
    print(f"{name:22s} | Best val MSE: {min(val):.4f} at epoch {best_ep}")

# ── Plots ──────────────────────────────────────────────────────────
x_plot = torch.linspace(-3.5, 3.5, 300).unsqueeze(1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Train vs val curves
for (name, _, __), col in zip(models_opts, colors):
    r = results[name]
    axes[0].plot(r['val'], color=col, lw=2, label=name, alpha=0.85)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val MSE')
axes[0].set_title('Validation Loss: All Regularization Methods')
axes[0].legend(fontsize=9); axes[0].set_ylim(0, 1)

# Final predictions
axes[1].scatter(X_tr.numpy(), y_tr.numpy(), color='#DD8452', s=40, zorder=5, label='Train data')
axes[1].plot(x_plot.numpy(), torch.sin(x_plot).numpy(), 'k--', lw=2, label='True sin(x)')
for (name, _, __), col in zip(models_opts, colors):
    m = results[name]['model']
    m.eval()
    with torch.no_grad():
        y_pred = m(x_plot).numpy()
    axes[1].plot(x_plot.numpy(), y_pred, color=col, lw=2, label=name, alpha=0.85)
axes[1].set_ylim(-2.5, 2.5); axes[1].set_xlabel('x'); axes[1].set_ylabel('y')
axes[1].set_title('Final Predictions: All Methods'); axes[1].legend(fontsize=8)

# Val loss comparison bar
final_vals = {name: min(results[name]['val']) for name, *_ in models_opts}
bars = axes[2].bar(list(final_vals.keys()), list(final_vals.values()),
                   color=colors, alpha=0.88)
axes[2].set_ylabel('Best Val MSE'); axes[2].set_title('Best Validation MSE Comparison')
axes[2].set_xticklabels(list(final_vals.keys()), rotation=15, ha='right', fontsize=9)
for bar, v in zip(bars, final_vals.values()):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Figure 9: All Regularization Techniques Combined', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---

## 10. Master Interview Q&A Cheatsheet

### Level 1 — Beginner

> **Q: What is overfitting and how do you detect it?**
> A: Overfitting = model performs well on training data but poorly on new data. Detect by comparing training vs validation loss: if training loss keeps decreasing while validation loss starts increasing — overfitting.

> **Q: Name 3 regularization techniques and what they do.**
> A: (1) L2/Weight Decay: penalises large weights, shrinks them toward 0. (2) Dropout: randomly zeros neurons during training, prevents co-adaptation. (3) Early stopping: stops training when validation loss stops improving.

> **Q: What does the dropout rate p mean?**
> A: Probability that any given neuron is set to 0 in a forward pass. p=0.5 means each neuron has 50% chance of being dropped. Higher p = more regularization.

> **Q: Why does adding more training data reduce overfitting?**
> A: More data gives the model a broader view of the true distribution, making it harder to memorise specific examples. Regularization is often a substitute for data when data is scarce.

---

### Level 2 — Mid-Level

> **Q: L1 vs L2 regularization — when would you choose each?**
> A: L2: most deep learning (built into AdamW, smooth gradients, shrinks all weights). L1: sparse problems — NLP bag-of-words, genomics (most features irrelevant, need exact zeros). L2 is the default; L1 when feature selection / sparsity is a goal.

> **Q: What is inverted dropout and why is it standard?**
> A: Scale dropped neurons' surviving counterparts by $1/(1-p)$ during training, so no scaling is needed at test time. Standard because inference code is simpler — don't need to remember to scale. PyTorch's `nn.Dropout` uses inverted dropout.

> **Q: Dropout vs Batch Normalization — are they redundant?**
> A: Mostly redundant — both regularize. BatchNorm's regularization comes from the noise of mini-batch statistics. Using both together often hurts performance (conflicting variance estimates). General rule: use BatchNorm for hidden layers; only add dropout for fully connected heads.

> **Q: What is the bias-variance tradeoff and how do regularization techniques affect it?**
> A: Bias = error from wrong assumptions (model too simple). Variance = error from sensitivity to training data (model too complex). Regularization reduces variance (prevents overfitting) at a small cost to bias. The right amount of regularization finds the sweet spot.

---

### Level 3 — Senior MLE / Staff Engineer

> **Q: A ResNet-50 is 3% below target accuracy. Walk me through your regularization checklist.**
> A: (1) Confirm it's overfitting (train acc >> val acc?) — if not, regularization isn't the problem. (2) Audit weight decay: AdamW with wd=1e-4 is standard for ResNets. (3) Check dropout: for CNNs, avoid in conv layers; only in FC head. (4) Add label smoothing=0.1. (5) Add random augmentation (CutMix, RandAugment). (6) Check batch size — larger batches may need stronger regularization. (7) Try stochastic depth (DropPath) — drop entire residual branches.

> **Q: Why does dropout interact poorly with Batch Normalization?**
> A: Dropout introduces random variance in layer outputs. BatchNorm normalises using batch statistics which now have this extra dropout noise. When switching to eval mode, dropout disappears but the running BatchNorm statistics were computed with it. This "variance shift" degrades test-time performance. Solution: don't use both on the same layer; dropout after BN (in FC heads) is safer.

> **Q: Explain the implicit regularization effect of mini-batch SGD.**
> A: Mini-batch gradient estimates are noisy. This noise acts as a stochastic perturbation that prevents sharp convergence — the optimizer can't perfectly fit the training loss minimum when each gradient is a noisy estimate. This noise correlates inversely with batch size. Smaller batches → more noise → more implicit regularization → often flatter minima. This is part of why very large batch training requires explicit regularization compensation (warmup, higher weight decay).

> **Q: Design a regularization strategy for a transformer-based model (e.g., BERT fine-tuning) showing signs of overfitting on a small dataset (2k examples).**
> A: (1) Strong weight decay (AdamW, wd=0.1 — higher than typical). (2) Dropout 0.1 in attention and FF layers (already in BERT, increase if needed). (3) Label smoothing 0.1 for classification head. (4) Early stopping (patience=5 on val loss). (5) Gradual unfreezing: freeze base layers first, fine-tune head, then progressively unfreeze. (6) Use smaller learning rate for base layers, larger for head. (7) Data augmentation if possible (paraphrase, back-translation for NLP). (8) Consider reducing model size — fine-tune BERT-small if overfitting is severe.
